In [ ]:
import pandas as pd
from utils import drop_rows, drop_columns
import pyarrow.parquet as pq

In [ ]:
file="../bronze/weather/weather.parquet"

In [ ]:
df = pd.read_parquet(file)
#df
#df.info()

metadata = pq.read_metadata(file)
print(metadata.metadata)

DELETE ALL THE DATA BEFORE AND AFTER THE YEAR 2013

In [ ]:
new_df= drop_rows(df,'date')
new_df
print(new_df.columns)
print(len(df))
print(len(new_df))

INVESTIGATE WHICH TYPE OF DATA INCLUDES THE OBJECT TYPE

In [ ]:
print(df.select_dtypes(include="object").dtypes)
for col in df.select_dtypes(include="object").columns:
    print(f"{col}: {df[col].dropna().head(1).tolist()}")

DELETE UNNECESSARY COLUMNS FOR THE ANALYSIS

In [ ]:
index_delete_columns=new_df.columns[16:59]
name_delete_columns=['latitude','longitude','elevation']

first_delete_columns=drop_columns(new_df, index_delete_columns)
second_delete_columns=drop_columns(first_delete_columns, name_delete_columns)

second_delete_columns.head()

GET ATTRIBUTES THAT INDICATED THE QUALITY OF INFORMATION

In [ ]:
# Select all attribute columns
attribute_columns = [
    col for col in second_delete_columns.columns
    if col.endswith("_attributes")
]

# Extract quality flags from all attribute columns
quality_flags = second_delete_columns[attribute_columns].apply(
    lambda col: col.str.split(",").str[1]
)

# Identify non-empty quality flags
has_quality_issue = quality_flags.notna() & quality_flags.ne("")

# Keep rows with at least one quality issue
not_empty_data = second_delete_columns[
    has_quality_issue.any(axis=1)
]

print("Rows with quality issues:", len(not_empty_data))
quality_flags["prcp_attributes"].value_counts(dropna=False)
has_quality_issue.sum().sort_values(ascending=False)

CHECK IF THE UNIT OF TEMPERATURES IS CELSIUS OR FAHRENHEIT

In [ ]:
print(second_delete_columns[["tmax", "tmin", "snow", "snwd"]].describe())

TRANSFORM THE TEMPERATURES IN CELSIUS WITH DECIMAL

In [ ]:
second_delete_columns["tmax"]=second_delete_columns["tmax"]/10
second_delete_columns["tmin"]=second_delete_columns["tmin"]/10

second_delete_columns

CHECK IF EXISTS DUPLICATED BY METHEOROLOGICAL STATION AND IF THERE IS MISSING DATES

In [ ]:
is_duplicated= second_delete_columns[second_delete_columns.duplicated(subset=["station","date"])]
print("Duplicated info: ",len(is_duplicated))
print(len(second_delete_columns))
print("First date:", second_delete_columns["date"].min())
print("Last date:", second_delete_columns["date"].max())

print("Unique dates:", second_delete_columns["date"].nunique())
print("Unique stations:", second_delete_columns["station"].nunique())

ANALYZE NULL VALUES

In [ ]:
# Calculate missing percentage
missing_percentage = (
    second_delete_columns.isna().mean() * 100
).sort_values(ascending=False)

missing_columns = second_delete_columns.loc[
    :, missing_percentage > 0.0
]
#missing_columns

empty_columns = missing_percentage[
    missing_percentage == 100
].index.tolist()

DELETE COLUMNS WITH EMPTY VALUES IN ALL CASES

In [ ]:
clean_dataset=drop_columns(second_delete_columns, empty_columns)
print(clean_dataset.columns)

In [26]:
invalid_temperature = clean_dataset[
    clean_dataset["tmax"] < clean_dataset["tmin"]
]

print("Invalid temperature records:", len(invalid_temperature))

Invalid temperature records: 0
